# Fashion Product Search: Text-Based Product Discovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SerendipityOneInc/ZooData-Notebook/blob/main/e-commerce-search/product_search.ipynb)

This notebook demonstrates how to use ZooData's **Fashion Product Search** API to discover fashion products across 200M+ items from major retailers using natural-language text queries.

### What you'll learn

1. **Basic text search** — find products with natural-language queries
2. **Brand filtering** — narrow results to specific brands
3. **Price range filtering** — search within a budget
4. **Retailer filtering** — allowlist or denylist specific retailer domains
5. **Building a product comparison table** — aggregate results across queries

### API Reference

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/openapi/v2/model/fashion-product-search` | POST | Search fashion products by text query |

## Setup

Sign up at [zoodata.ai/register](https://zoodata.ai/register) to get your API key — every new account includes 1,000 free credits to get started. Need more? Check out our [credit packages](https://zoodata.ai/pricing) for higher volumes.

In [ ]:
!pip install -q httpx Pillow matplotlib

In [ ]:
import getpass
import os

if "ZOODATA_API_KEY" not in os.environ:
    os.environ["ZOODATA_API_KEY"] = getpass.getpass("Enter your ZooData API key (hms_live_...): ")

API_KEY = os.environ["ZOODATA_API_KEY"]
BASE_URL = "https://api.zoodata.ai/openapi/v2"

In [ ]:
import httpx
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

client = httpx.Client(
    base_url=BASE_URL,
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30.0,
)


def fashion_search(query: str, top_k: int = 10, **kwargs) -> list[dict]:
    """Search fashion products by text query."""
    payload = {"query": query, "topK": top_k, **kwargs}
    resp = client.post("/model/fashion-product-search", json=payload)
    resp.raise_for_status()
    return resp.json()["data"]["products"]


def load_image(url: str) -> Image.Image:
    """Download an image from URL."""
    r = httpx.get(url, timeout=10, follow_redirects=True)
    return Image.open(BytesIO(r.content))


def show_products(products: list[dict], max_cols: int = 5):
    """Display product images in a grid."""
    n = min(len(products), max_cols)
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, p in zip(axes, products[:n]):
        try:
            ax.imshow(load_image(p.get("imageUrl", "")))
        except Exception:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center")
        title = (p.get("title") or "Unknown")[:40]
        price = p.get("price")
        brand = p.get("brand") or ""
        label = f"{title}...\n{brand}" + (f" | ${price:.2f}" if price else "")
        ax.set_title(label, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

---
## Task 1: Basic Text Search

The simplest use case — type a natural-language query and get matching products from 200M+ fashion items across major retailers.

In [ ]:
# Simple natural-language search
results = fashion_search("women red floral midi dress", top_k=5)

print(f"Found {len(results)} products\n")
for p in results:
    print(f"  {p.get('title', 'N/A')[:60]}")
    print(f"    Brand: {p.get('brand', '-')} | Price: ${p.get('price', 0):.2f} | Site: {p.get('site', '-')}")
    print()

show_products(results)

### Understanding the Product Object

Each product in the response is a rich, structured object — not raw retailer HTML. Every field has been through a multi-stage data pipeline:

1. **Raw collection** from retailer product pages across 200M+ items
2. **LLM-powered extraction & normalization** — brand names, categories, and product tags are parsed and standardized by large language models
3. **Human verification** — extracted metadata is reviewed and corrected by human annotators for accuracy

This means fields like `brand`, `category`, and `site` are **clean, consistent, and ready for analytics** — no regex hacking or ad-hoc parsing required.

#### Key highlights

| Field | Why it matters |
|-------|---------------|
| `imageUrl` | **Background-removed product image** — a clean shop figure with the background stripped, ideal for catalogs, visual search, and embedding generation |
| `brand` | LLM-extracted and human-verified brand name (e.g. `"Nike"`, `"Coach"`) — normalized across retailers that format brands differently |
| `category` | Standardized product category tag (e.g. `"Dresses"`, `"Shoes"`, `"Bags"`) — consistent taxonomy across all 200M+ products |
| `site` | Retailer domain (e.g. `"nordstrom.com"`, `"farfetch.com"`) — enables cross-retailer price comparison and market analysis |
| `price` | Price in USD, converted and normalized from the source retailer |
| `score` | **Semantic search relevance score** — computed by fashion-specific embedding models measuring how well the product matches your text query. Higher = better match. This is a search-engine metric, not user-generated |
| `link` | Direct link to the product page on the retailer's site |
| `rating` | **Customer review rating (0–5 scale)** — the product's average star rating from the retailer's site (e.g. 4.3 out of 5 on Nordstrom). `null` when the retailer doesn't provide ratings or the product has no reviews |
| `ratingsTotal` | Total number of customer reviews behind the `rating` average — useful for confidence weighting (a 4.5 with 2,000 reviews is more reliable than a 5.0 with 3) |

> **`score` vs `rating` — a common confusion:** `score` tells you how well the product matches *your search query* (search relevance). `rating` tells you how well *customers liked the product* (user satisfaction). They measure completely different things.

Let's inspect the full structure of a single product:

In [ ]:
# Inspect the full structure of a single product
import json

sample = results[0]
print("Full product object returned by the API:\n")
print(json.dumps(sample, indent=2, default=str))

print("\n" + "=" * 70)
print("FIELD-BY-FIELD REFERENCE")
print("=" * 70)

field_docs = {
    "productId": (
        "Unique product identifier",
        "Stable ID for tracking the same product across API calls"
    ),
    "title": (
        "Full product title from the retailer",
        "Original listing title — useful for display and keyword analysis"
    ),
    "brand": (
        "Brand name (LLM-extracted, human-verified)",
        "Normalized across retailers — e.g. 'Ralph Lauren' not 'RALPH LAUREN CORPORATION'"
    ),
    "category": (
        "Standardized product category tag",
        "Consistent taxonomy (e.g. 'Dresses', 'Shoes', 'Bags') across all 200M+ products"
    ),
    "price": (
        "Price in USD",
        "Converted and normalized from the retailer's native currency"
    ),
    "currencySymbol": (
        "Currency symbol (e.g. '$')",
        "Accompanies the price field for display formatting"
    ),
    "imageUrl": (
        "Background-removed product image",
        "Clean shop figure with background stripped — ideal for catalogs, visual search, and embeddings"
    ),
    "link": (
        "Direct link to the product page",
        "Deep link to the retailer's product listing"
    ),
    "site": (
        "Retailer domain (e.g. 'nordstrom.com')",
        "Enables cross-retailer comparison, affiliate routing, and market coverage analysis"
    ),
    "score": (
        "Semantic search relevance score (search-engine metric, NOT user-generated)",
        "Computed by fashion-specific embedding models — measures how well the product matches your text query. Higher = better match"
    ),
    "rating": (
        "Customer review rating on a 0–5 scale (user-generated, NOT a search metric)",
        "Average star rating from the retailer's site (e.g. 4.3/5 on Nordstrom). null when unavailable"
    ),
    "ratingsTotal": (
        "Total number of customer reviews behind the rating",
        "Useful for confidence weighting — a 4.5 with 2000 reviews is more reliable than a 5.0 with 3"
    ),
}

for field, (short_desc, detail) in field_docs.items():
    value = sample.get(field)
    print(f"\n  📌 {field}")
    print(f"     {short_desc}")
    print(f"     → {detail}")
    print(f"     Example value: {repr(value)}")

print("\n" + "=" * 70)
print("💡 score vs rating")
print("=" * 70)
print("  score  = how well the product matches YOUR SEARCH QUERY (search relevance)")
print("  rating = how well CUSTOMERS liked the product (user satisfaction)")
print("  These measure completely different things!")

#### Background-Removed Product Images

The `imageUrl` field returns a **clean, background-removed shop figure** — not the raw retailer photo with cluttered backgrounds, watermarks, or lifestyle staging. This makes the images immediately usable for:

- **Visual search & similarity** — feed directly into embedding models without preprocessing
- **Product catalogs & lookbooks** — consistent, clean presentation across retailers
- **AI/ML pipelines** — no need for background removal models like rembg or SAM

In [ ]:
# Display background-removed product images side by side
fig, axes = plt.subplots(1, min(len(results), 5), figsize=(18, 4))
if not isinstance(axes, list):
    axes = list(axes) if hasattr(axes, '__iter__') else [axes]

for ax, p in zip(axes, results[:5]):
    try:
        img = load_image(p.get("imageUrl", ""))
        ax.imshow(img)
    except Exception:
        ax.text(0.5, 0.5, "N/A", ha="center", va="center", fontsize=14)
    ax.set_title(
        f"{(p.get('brand') or 'Unknown'):}\n{(p.get('category') or '-')} | {p.get('site', '')}",
        fontsize=9, fontweight="bold",
    )
    ax.axis("off")

fig.suptitle("Background-Removed Product Images — clean, consistent, ready for ML", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Try different query styles
queries = [
    "black leather ankle boots",
    "men slim fit navy blazer",
    "women brown suede tote under 300",
    "summer linen shorts casual",
]

for q in queries:
    products = fashion_search(q, top_k=3)
    print(f"\n\"{q}\" → {len(products)} results")
    for p in products:
        print(f"  {p.get('title', '')[:55]} | ${p.get('price', 0):.2f} | {p.get('site', '')}")

---
## Task 2: Brand Filtering

Narrow results to a specific brand. Useful for brand-specific product research, competitive benchmarking, or building brand-focused shopping assistants.

In [ ]:
# Search for handbags from a specific brand
coach_bags = fashion_search("leather tote bag", top_k=5, brand="Coach")

print(f"Coach leather tote bags: {len(coach_bags)} results\n")
for p in coach_bags:
    print(f"  {p.get('title', '')[:55]}")
    print(f"    ${p.get('price', 0):.2f} | {p.get('site', '')}")

show_products(coach_bags)

In [ ]:
# Compare the same product category across brands
brands = ["Nike", "Adidas", "New Balance"]
brand_results = {}

for brand in brands:
    products = fashion_search("running shoes", top_k=5, brand=brand)
    brand_results[brand] = products
    avg_price = sum(p.get("price", 0) for p in products) / max(len(products), 1)
    print(f"{brand}: {len(products)} results, avg price ${avg_price:.2f}")

# Show top result from each brand side by side
top_picks = [products[0] for products in brand_results.values() if products]
show_products(top_picks)

---
## Task 3: Price Range Filtering

Filter results by price range (in USD). Useful for budget-aware recommendations or tiered product catalogs.

In [ ]:
# Search within a budget
budget_dresses = fashion_search(
    "elegant evening dress",
    top_k=5,
    priceMin=50,
    priceMax=200,
)

print(f"Evening dresses $50-$200: {len(budget_dresses)} results\n")
for p in budget_dresses:
    print(f"  ${p.get('price', 0):>7.2f} | {p.get('title', '')[:50]} | {p.get('site', '')}")

show_products(budget_dresses)

In [ ]:
# Price tier analysis: same query across budget / mid / premium
tiers = [
    ("Budget",  0, 50),
    ("Mid-range", 50, 200),
    ("Premium", 200, 1000),
]

print("Price tier analysis for 'women handbag':\n")
for label, lo, hi in tiers:
    products = fashion_search("women handbag", top_k=10, priceMin=lo, priceMax=hi)
    prices = [p.get("price", 0) for p in products if p.get("price")]
    if prices:
        print(f"  {label:>10} (${lo}-${hi}): {len(products)} results | "
              f"avg ${sum(prices)/len(prices):.2f} | "
              f"range ${min(prices):.2f}-${max(prices):.2f}")
    else:
        print(f"  {label:>10} (${lo}-${hi}): no results")

---
## Task 4: Retailer Filtering

Use `sites` (allowlist) or `excludeSites` (denylist) to control which retailer domains appear in results. Useful for affiliate programs, marketplace comparison, or excluding resale platforms.

In [ ]:
# Only search on specific retailers
luxury_sites = fashion_search(
    "designer sunglasses",
    top_k=5,
    sites=["farfetch.com", "nordstrom.com", "net-a-porter.com"],
)

print(f"Designer sunglasses from luxury retailers: {len(luxury_sites)} results\n")
for p in luxury_sites:
    print(f"  {p.get('site', ''):>20} | ${p.get('price', 0):.2f} | {p.get('title', '')[:45]}")

In [ ]:
# Exclude resale / secondhand platforms
new_only = fashion_search(
    "vintage style denim jacket",
    top_k=5,
    excludeSites=["poshmark.com", "therealreal.com", "ebay.com"],
)

print(f"Denim jackets (excl. resale): {len(new_only)} results\n")
for p in new_only:
    print(f"  {p.get('site', ''):>20} | ${p.get('price', 0):.2f} | {p.get('title', '')[:45]}")

show_products(new_only)

---
## Summary

This notebook demonstrated how to use ZooData's Fashion Product Search API for text-based fashion product discovery.

### API Used

| Endpoint | Purpose | Cost |
|----------|---------|------|
| `POST /openapi/v2/model/fashion-product-search` | Text-based fashion product search | 1 credit/request |

### Product Response Fields

Each product object is **LLM-extracted and human-verified** — clean, normalized, and ready for analytics:

| Field | Description | Example |
|-------|-------------|---------|
| `productId` | Unique product identifier | `"abc123"` |
| `title` | Full product title from the retailer | `"Women's Red Floral Midi Dress"` |
| `brand` | Normalized brand name (LLM-extracted, human-verified) | `"Nike"`, `"Coach"` |
| `category` | Standardized product category tag | `"Dresses"`, `"Shoes"`, `"Bags"` |
| `price` | Price in USD | `129.99` |
| `currencySymbol` | Currency symbol | `"$"` |
| `imageUrl` | **Background-removed** clean product image | URL to shop figure |
| `link` | Direct link to retailer product page | URL |
| `site` | Retailer domain | `"nordstrom.com"`, `"farfetch.com"` |
| `score` | **Semantic search relevance** — how well the product matches your query (search-engine metric, higher = better match) | `0.87` |
| `rating` | **Customer review rating** — average star rating from the retailer's site (0–5, null if unavailable) | `4.5` |
| `ratingsTotal` | Total number of customer reviews behind the rating | `2847` |

> **`score` vs `rating`:** `score` = search relevance (how well it matches your query). `rating` = user satisfaction (how well customers liked it). Different metrics for different purposes.

### Key Parameters

| Parameter | Description |
|-----------|-------------|
| `query` | Natural-language search query |
| `topK` | Max results (1-50) |
| `brand` | Filter by brand name |
| `priceMin` / `priceMax` | Price range in USD |
| `sites` | Retailer domain allowlist |
| `excludeSites` | Retailer domain denylist |

### Next Steps

- Try the [Fashion Image Search](https://colab.research.google.com/github/SerendipityOneInc/ZooData-Notebook/blob/main/e-commerce-search/image_search.ipynb) notebook for visual similarity search
- Combine text search with FashionSigLIP2 embeddings for custom reranking
- Sign up at [zoodata.ai](https://zoodata.ai) and explore our [credit packages](https://zoodata.ai/pricing) for production workloads

In [ ]:
# Build a comparison dataset across categories
categories = {
    "Dresses": "women summer dress",
    "Sneakers": "casual white sneakers",
    "Bags": "leather crossbody bag",
    "Jackets": "men bomber jacket",
}

all_products = []
for cat_name, query in categories.items():
    products = fashion_search(query, top_k=10)
    for p in products:
        all_products.append({
            "category": cat_name,
            "title": (p.get("title") or "")[:60],
            "brand": p.get("brand", "-"),
            "price": p.get("price", 0),
            "site": p.get("site", "-"),
            "rating": p.get("rating"),
            "score": p.get("score"),
        })

print(f"Total products collected: {len(all_products)}")

In [ ]:
---
## Summary

This notebook demonstrated how to use ZooData's Fashion Product Search API for text-based fashion product discovery.

### API Used

| Endpoint | Purpose | Cost |
|----------|---------|------|
| `POST /openapi/v2/model/fashion-product-search` | Text-based fashion product search | 1 credit/request |

### Product Response Fields

Each product object includes: `productId`, `title`, `brand`, `price`, `currencySymbol`, `imageUrl`, `link`, `site`, `score`, `rating`, `ratingsTotal`, and `category` — giving you everything needed for product comparison, recommendation, and analytics workflows.

### Key Parameters

| Parameter | Description |
|-----------|-------------|
| `query` | Natural-language search query |
| `topK` | Max results (1-50) |
| `brand` | Filter by brand name |
| `priceMin` / `priceMax` | Price range in USD |
| `sites` | Retailer domain allowlist |
| `excludeSites` | Retailer domain denylist |

### Next Steps

- Try the [Fashion Image Search](https://colab.research.google.com/github/SerendipityOneInc/ZooData-Notebook/blob/main/e-commerce-search/image_search.ipynb) notebook for visual similarity search
- Combine text search with FashionSigLIP2 embeddings for custom reranking
- Sign up at [zoodata.ai](https://zoodata.ai) and explore our [credit packages](https://zoodata.ai/pricing) for production workloads

In [ ]:
# Visualize price distribution by category
fig, ax = plt.subplots(figsize=(10, 5))

cat_names = list(categories.keys())
cat_prices = []
for cat in cat_names:
    prices = [p["price"] for p in all_products if p["category"] == cat and p["price"]]
    cat_prices.append(prices)

ax.boxplot(cat_prices, labels=cat_names)
ax.set_ylabel("Price (USD)")
ax.set_title("Price Distribution by Fashion Category")
plt.tight_layout()
plt.show()

---
## Summary

This notebook demonstrated how to use ZooData's Fashion Product Search API for text-based fashion product discovery.

### API Used

| Endpoint | Purpose | Cost |
|----------|---------|------|
| `POST /openapi/v2/model/fashion-product-search` | Text-based fashion product search | 1 credit/request |

### Key Parameters

| Parameter | Description |
|-----------|-------------|
| `query` | Natural-language search query |
| `topK` | Max results (1-50) |
| `brand` | Filter by brand name |
| `priceMin` / `priceMax` | Price range in USD |
| `sites` | Retailer domain allowlist |
| `excludeSites` | Retailer domain denylist |

### Next Steps

- Try the [Fashion Image Search](https://colab.research.google.com/github/SerendipityOneInc/ZooData-Notebook/blob/main/e-commerce-search/image_search.ipynb) notebook for visual similarity search
- Combine text search with FashionSigLIP2 embeddings for custom reranking
- Get your API key at [zoodata.ai](https://zoodata.ai)